# BirdCLEF+ 2026: Training Data Notebook

This notebook loads and checks the **training data** that the EoS.9 ensemble relies on, and reproduces the way EoS.9 prepares it. It covers:

1. The label space (234 classes) and the taxonomy
2. Focal recordings (`train.csv`)
3. Labelled soundscapes (`train_soundscapes_labels.csv`), including the "fully labelled" files the sequence models train on
4. How well the two sources cover the classes
5. Cross-validation folds (as in the Model_1 SED code)
6. Simple site and hour prior tables
7. Optional checks: the cached Perch embeddings and a spot check of the audio files

**Requirements.** Only `pandas`, `numpy`, `matplotlib` and `scikit-learn` are needed for the core sections. `soundfile` is used for the optional audio check. On Kaggle, attach the BirdCLEF+ 2026 competition data. Elsewhere, set the `BIRDCLEF_DIR` environment variable to the folder that contains `sample_submission.csv` and `taxonomy.csv`.

The reference numbers quoted in the notes (59 files, 708 windows, 71 active classes) come from the saved run logs of EoS.9. Your numbers may differ if the data version differs.

In [ ]:
import os, re, json, ast, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# ---- constants (same values the EoS.9 notebook uses) ----
SR         = 32_000                 # sample rate in Hz
WINDOW_SEC = 5                      # one prediction window
FILE_SEC   = 60                     # one soundscape file
N_WINDOWS  = FILE_SEC // WINDOW_SEC # 12 windows per file
SEED       = 42

# File names look like BC2026_Train_0001_S08_20250606_030007.ogg
FNAME_RE = re.compile(r"BC2026_(Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"split": None, "file_id": None, "site": "unknown", "date": pd.NaT,
                "time_utc": None, "hour_utc": -1, "month": -1}
    split, file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {"split": split, "file_id": file_id, "site": site, "date": dt, "time_utc": hms,
            "hour_utc": int(hms[:2]), "month": int(dt.month) if pd.notna(dt) else -1}

def find_competition_dir():
    # Set BIRDCLEF_DIR to point at the data when running outside Kaggle.
    candidates = []
    if os.environ.get("BIRDCLEF_DIR"):
        candidates.append(Path(os.environ["BIRDCLEF_DIR"]))
    candidates += [Path("/kaggle/input/competitions/birdclef-2026"), Path("/kaggle/input/birdclef-2026")]
    for p in candidates:
        if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        for p in root.rglob("sample_submission.csv"):
            if (p.parent / "taxonomy.csv").exists():
                return p.parent
    raise FileNotFoundError("BirdCLEF competition data not found. Attach the competition data "
                            "or set the BIRDCLEF_DIR environment variable.")

BASE = find_competition_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Competition data:", BASE)
print("Contents        :", sorted(p.name for p in BASE.iterdir())[:20])
print("Outputs go to   :", OUT_DIR.resolve())

In [ ]:
MIN_SAMPLE = 20   # species with fewer focal recordings are treated as "rare" (EoS.9 upsamples them to 20)
N_FOLDS    = 5

## 1. Label space

`sample_submission.csv` defines the order of the label columns, so it is the source of truth for the class list. `taxonomy.csv` adds the scientific name and the taxon (`class_name`) of each class.

In [ ]:
sample_sub = pd.read_csv(BASE / "sample_submission.csv", nrows=5)
PRIMARY_LABELS = [c for c in sample_sub.columns if c != "row_id"]
NUM_CLASSES = len(PRIMARY_LABELS)
LABEL2IDX = {l: i for i, l in enumerate(PRIMARY_LABELS)}

taxonomy = pd.read_csv(BASE / "taxonomy.csv")
taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)

keep = [c for c in ["primary_label", "scientific_name", "common_name", "class_name"] if c in taxonomy.columns]
label_df = pd.DataFrame({"primary_label": PRIMARY_LABELS}).merge(taxonomy[keep], on="primary_label", how="left")
label_df["taxon"] = label_df["class_name"].fillna("Unknown") if "class_name" in label_df else "Unknown"
label_df["contains_son"]  = label_df["primary_label"].str.contains("son")
label_df["is_sonotype"]   = label_df["primary_label"].str.match(r"^\d+son\d+$")

print(f"Classes in sample_submission: {NUM_CLASSES}")
print(f"Classes missing from taxonomy: {int(label_df['scientific_name'].isna().sum()) if 'scientific_name' in label_df else 'n/a'}")
print(f"Labels containing 'son': {int(label_df['contains_son'].sum())}  |  matching <digits>son<digits>: {int(label_df['is_sonotype'].sum())}")

taxon_counts = label_df["taxon"].value_counts().rename_axis("taxon").reset_index(name="n_classes")
taxon_counts["share_%"] = (100 * taxon_counts["n_classes"] / NUM_CLASSES).round(1)
display(taxon_counts)

ax = taxon_counts.sort_values("n_classes").plot.barh(x="taxon", y="n_classes", legend=False, color="#2A7F8E", figsize=(6, 3))
ax.set_xlabel("Number of classes"); ax.set_title("Label space by taxon")
plt.tight_layout(); plt.show()

## 2. Focal recordings (`train.csv`)

Focal recordings are single-species clips from a public archive. EoS.9 uses them only to train the SED model (Model_1). Each row has a primary label and, usually, a list of secondary labels.

In [ ]:
train_df = pd.read_csv(BASE / "train.csv")
train_df["primary_label"] = train_df["primary_label"].astype(str)
print("Columns:", list(train_df.columns))

def parse_secondary(x):
    if pd.isna(x):
        return []
    try:
        v = ast.literal_eval(x) if isinstance(x, str) else x
        return [str(t) for t in v] if isinstance(v, (list, tuple)) else []
    except Exception:
        return [t.strip() for t in str(x).strip("[]").replace("'", "").split(",") if t.strip()]

in_space = train_df["primary_label"].isin(LABEL2IDX)
print(f"Recordings: {len(train_df):,}  |  primary label in label space: {int(in_space.sum()):,}  |  dropped: {int((~in_space).sum()):,}")

focal = train_df[in_space].copy().reset_index(drop=True)
if "secondary_labels" in focal:
    focal["secondary_list"] = focal["secondary_labels"].apply(parse_secondary)
    focal["n_secondary_in_space"] = focal["secondary_list"].apply(lambda L: sum(t in LABEL2IDX for t in L))
    print(f"Recordings with at least one secondary label in the label space: {(focal['n_secondary_in_space'] > 0).mean():.1%}")

focal_counts = focal["primary_label"].value_counts().reindex(PRIMARY_LABELS, fill_value=0)
summary = {
    "classes with >=1 recording": int((focal_counts > 0).sum()),
    "classes with no recording":  int((focal_counts == 0).sum()),
    f"rare classes (1..{MIN_SAMPLE-1} recordings)": int(((focal_counts > 0) & (focal_counts < MIN_SAMPLE)).sum()),
    "median recordings per class": float(focal_counts.median()),
    "max recordings per class": int(focal_counts.max()),
}
display(pd.Series(summary, name="value").to_frame())
if "rating" in focal:
    print("Rating summary:"); display(focal["rating"].describe().round(2).to_frame().T)

display(focal_counts.sort_values(ascending=False).head(10).rename("recordings").to_frame())

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(NUM_CLASSES), np.sort(focal_counts.values)[::-1], color="#1F3A5F", width=1.0)
ax.axhline(MIN_SAMPLE, color="#B5533C", ls="--", lw=1, label=f"rare threshold ({MIN_SAMPLE})")
ax.set_yscale("symlog"); ax.set_xlabel("Classes (sorted)"); ax.set_ylabel("Focal recordings")
ax.set_title("Focal recordings per class"); ax.legend(); plt.tight_layout(); plt.show()

## 3. Labelled soundscapes (`train_soundscapes_labels.csv`)

These are expert labels for 5-second windows of 60-second field recordings. EoS.9 processes them as follows:

1. Drop duplicated rows.
2. Merge the labels of each `(filename, start, end)` window into one set (labels are semicolon-separated).
3. Build a `row_id` = file name without `.ogg` + `_` + end second.
4. Parse site, date, UTC time and hour from the file name.
5. Build a multi-hot label matrix.
6. Keep only **fully labelled files** (all 12 windows present). Their windows are called the *trusted* windows.

In [ ]:
labels_raw = pd.read_csv(BASE / "train_soundscapes_labels.csv").drop_duplicates()
print(f"Label rows after dropping duplicates: {len(labels_raw):,}")

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

sc = (labels_raw.groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels).reset_index(name="label_list"))
sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)
sc = pd.concat([sc, sc["filename"].apply(parse_fname).apply(pd.Series)], axis=1)

Y_SC = np.zeros((len(sc), NUM_CLASSES), dtype=np.uint8)
unknown_labels = set()
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in LABEL2IDX:
            Y_SC[i, LABEL2IDX[lbl]] = 1
        else:
            unknown_labels.add(lbl)

windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)

full_rows = sc[sc["fully_labeled"]].sort_values(["filename", "end_sec"]).reset_index(drop=False)
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

print(f"Unique labelled windows        : {len(sc):,}")
print(f"Labelled files                 : {sc['filename'].nunique():,}")
print(f"Fully labelled files (12 wins) : {len(full_files):,}")
print(f"Trusted windows                : {len(full_rows):,}")
print(f"Labels not in the label space  : {sorted(unknown_labels) if unknown_labels else 'none'}")

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
windows_per_file.value_counts().sort_index().plot.bar(ax=axes[0], color="#2A7F8E")
axes[0].set_xlabel("Labelled windows in a file"); axes[0].set_ylabel("Files"); axes[0].set_title("Windows per file")
pd.Series(Y_SC.sum(axis=1)).value_counts().sort_index().plot.bar(ax=axes[1], color="#1F3A5F")
axes[1].set_xlabel("Positive labels in a window"); axes[1].set_ylabel("Windows"); axes[1].set_title("Labels per window")
plt.tight_layout(); plt.show()

In [ ]:
# Positives per class in the trusted windows, joined with the taxonomy
pos_full = pd.Series(Y_FULL.sum(axis=0), index=PRIMARY_LABELS, name="trusted_positive_windows")
class_stats = label_df.set_index("primary_label")[["taxon"]].join(pos_full)
class_stats["active"] = class_stats["trusted_positive_windows"] > 0

print(f"Active classes (>=1 positive in trusted windows): {int(class_stats['active'].sum())} of {NUM_CLASSES}")
print(f"Classes with no positive: {int((~class_stats['active']).sum())} ({(~class_stats['active']).mean():.0%})")

by_taxon = class_stats.groupby("taxon").agg(n_classes=("active", "size"), n_active=("active", "sum"),
                                            positive_windows=("trusted_positive_windows", "sum"))
by_taxon["active_%"] = (100 * by_taxon["n_active"] / by_taxon["n_classes"]).round(1)
display(by_taxon)

top = class_stats.sort_values("trusted_positive_windows", ascending=False).head(15)
ax = top["trusted_positive_windows"][::-1].plot.barh(figsize=(6, 4), color="#2A7F8E")
ax.set_xlabel("Positive windows"); ax.set_title("Most frequent classes in trusted windows")
plt.tight_layout(); plt.show()

In [ ]:
# Sites, hours and dates of the labelled files
files_meta = (sc.drop_duplicates("filename")[["filename", "site", "date", "hour_utc", "month", "fully_labeled"]]
              .reset_index(drop=True))
site_tbl = (files_meta.groupby("site").agg(files=("filename", "size"), fully_labeled=("fully_labeled", "sum"),
                                           first_date=("date", "min"), last_date=("date", "max")).reset_index())
display(site_tbl)
print("Sites among fully labelled files:", sorted(files_meta.loc[files_meta["fully_labeled"], "site"].unique()))
print("S22 is flagged in the Model_1 code as a site with known label noise.")

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
files_meta["site"].value_counts().plot.bar(ax=axes[0], color="#1F3A5F")
axes[0].set_title("Labelled files per site"); axes[0].set_ylabel("Files")
files_meta["hour_utc"].value_counts().reindex(range(24), fill_value=0).plot.bar(ax=axes[1], color="#2A7F8E")
axes[1].set_title("Labelled files per UTC hour"); axes[1].set_xlabel("hour_utc")
plt.tight_layout(); plt.show()

## 4. Coverage of the label space by both sources

A class can have focal recordings, trusted soundscape positives, both, or neither. Classes with neither depend on the pretrained Perch model, taxonomic proxies and priors.

In [ ]:
cov = class_stats.copy()
cov["focal_recordings"] = focal_counts
cov["has_focal"] = cov["focal_recordings"] > 0
cov["has_soundscape"] = cov["active"]
cov["coverage"] = np.select(
    [cov.has_focal & cov.has_soundscape, cov.has_focal & ~cov.has_soundscape, ~cov.has_focal & cov.has_soundscape],
    ["both", "focal only", "soundscape only"], default="neither")

display(cov["coverage"].value_counts().rename_axis("coverage").reset_index(name="n_classes"))
display(pd.crosstab(cov["taxon"], cov["coverage"]))
print("Classes with neither source (first 20):")
display(cov[cov["coverage"] == "neither"].head(20))
cov.to_csv(OUT_DIR / "class_coverage.csv")

## 5. Cross-validation folds

The Model_1 code uses two schemes:

* **Focal recordings:** stratified 5-fold split by species.
* **Soundscapes:** 5-fold split grouped by file, so all windows of a file stay together. All labelled files take part, and the S22 site is only excluded from the primary metric.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold

# --- focal: stratified by species ---
focal_f = focal.copy()
focal_f["fold"] = -1
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        for f, (_, val_idx) in enumerate(skf.split(focal_f, focal_f["primary_label"])):
            focal_f.loc[val_idx, "fold"] = f
    except ValueError as e:
        print("StratifiedKFold not possible:", e)
if caught:
    print("Note:", str(caught[0].message)[:160])
print("Focal recordings per fold:"); display(focal_f["fold"].value_counts().sort_index().to_frame("recordings").T)

# --- soundscapes: grouped by file ---
sc_files = sc[["filename", "site"]].drop_duplicates().reset_index(drop=True)
k = min(N_FOLDS, len(sc_files))
sc_files["fold"] = -1
for f, (_, val_idx) in enumerate(GroupKFold(n_splits=k).split(sc_files, groups=sc_files["filename"])):
    sc_files.loc[val_idx, "fold"] = f
sc["fold"] = sc["filename"].map(dict(zip(sc_files["filename"], sc_files["fold"]))).astype(int)

print("Soundscape windows per fold:"); display(sc["fold"].value_counts().sort_index().to_frame("windows").T)
folds_per_class = pd.Series([len(set(sc.loc[Y_SC[:, j] == 1, "fold"])) for j in range(NUM_CLASSES)], index=PRIMARY_LABELS)
print(f"Classes with positives in >=2 folds (usable for validation): {int((folds_per_class >= 2).sum())}")
print(f"Classes with positives in exactly 1 fold: {int((folds_per_class == 1).sum())}")

## 6. Site and hour prior tables

EoS.9 fuses model scores with priors: how often each class is positive at a site, at an hour of the day, and at a site-hour pair, all computed from the trusted windows. The tables below are the plain frequency version.

In [ ]:
full_meta = full_rows[["filename", "site", "hour_utc"]].reset_index(drop=True)
global_p = Y_FULL.mean(axis=0)

def freq_table(keys):
    g = full_meta.groupby(keys).indices
    idx = list(g.keys())
    p = np.vstack([Y_FULL[g[k]].mean(axis=0) for k in idx]) if idx else np.zeros((0, NUM_CLASSES))
    n = np.array([len(g[k]) for k in idx])
    return idx, p, n

site_keys, site_p, site_n = freq_table("site")
hour_keys, hour_p, hour_n = freq_table("hour_utc")
print(f"Global prior: mean positive rate {global_p.mean():.4f}, max {global_p.max():.3f}")
print(f"Site table: {site_p.shape}  |  Hour table: {hour_p.shape}")
display(pd.DataFrame({"site": site_keys, "windows": site_n}))
display(pd.DataFrame({"hour_utc": hour_keys, "windows": hour_n}).T)
print("Hours with no trusted windows:", sorted(set(range(24)) - set(hour_keys)))

## 7. Optional: check the cached Perch embeddings

EoS.9 (Models 51 and 74) reads a precomputed Perch v2 cache made of `full_perch_meta.parquet` and `full_perch_arrays.npz`. This cell looks for it under `/kaggle/input` and checks that its shapes match the trusted windows. It is skipped if the cache is not attached.

In [ ]:
cache_meta = cache_npz = None
root = Path("/kaggle/input")
if root.exists():
    for m in sorted(root.rglob("full_perch_meta.parquet")):
        if (m.parent / "full_perch_arrays.npz").exists():
            cache_meta, cache_npz = m, m.parent / "full_perch_arrays.npz"; break

if cache_meta is None:
    print("Perch cache not found, skipping. (Attach a dataset containing full_perch_meta.parquet and full_perch_arrays.npz.)")
else:
    print("Using cache:", cache_meta.parent)
    meta = pd.read_parquet(cache_meta)
    arr = np.load(cache_npz, allow_pickle=True)
    print("Arrays in npz:", {k: getattr(arr[k], "shape", None) for k in arr.files})
    scores = next((arr[k] for k in ["scores", "sc", "logits"] if k in arr.files), None)
    embs   = next((arr[k] for k in ["embs", "emb", "embeddings"] if k in arr.files), None)
    checks = {
        "meta rows == trusted windows": len(meta) == len(full_rows),
        "scores shape": None if scores is None else scores.shape == (len(full_rows), NUM_CLASSES),
        "embeddings shape": None if embs is None else embs.shape == (len(full_rows), 1536),
    }
    if "primary_labels" in arr.files:
        checks["label order matches sample_submission"] = arr["primary_labels"].tolist() == PRIMARY_LABELS
    display(pd.Series(checks, name="ok").to_frame())

## 8. Optional: audio spot check

Reads only the headers (no decoding) of a few files to confirm sample rate and duration.

In [ ]:
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("soundfile is not installed, skipping.")

if sf is not None:
    rows = []
    sc_dir = BASE / "train_soundscapes"
    for fn in (full_files[:5] if full_files else []):
        p = sc_dir / fn
        if p.exists():
            i = sf.info(str(p)); rows.append(("soundscape", fn, i.samplerate, i.channels, round(i.frames / i.samplerate, 1)))
    audio_dir = BASE / "train_audio"
    if audio_dir.exists() and "filename" in focal:
        for fn in focal["filename"].head(50):
            p = audio_dir / fn
            if p.exists():
                i = sf.info(str(p)); rows.append(("focal", fn, i.samplerate, i.channels, round(i.frames / i.samplerate, 1)))
            if sum(r[0] == "focal" for r in rows) >= 5:
                break
    if rows:
        display(pd.DataFrame(rows, columns=["kind", "file", "sample_rate", "channels", "duration_s"]))
    else:
        print("No audio files found to inspect.")

## 9. Summary and export

The table compares this run with the reference values printed in the EoS.9 logs. A mismatch usually means a different data version rather than an error.

In [ ]:
REFERENCE = {"classes": 234, "fully_labelled_files": 59, "trusted_windows": 708, "active_classes": 71}
summary = {
    "classes": NUM_CLASSES,
    "focal_recordings_in_label_space": int(len(focal)),
    "labelled_windows": int(len(sc)),
    "labelled_files": int(sc["filename"].nunique()),
    "fully_labelled_files": len(full_files),
    "trusted_windows": int(len(full_rows)),
    "active_classes": int(class_stats["active"].sum()),
    "classes_without_positive": int((~class_stats["active"]).sum()),
    "sites_in_trusted_files": sorted(files_meta.loc[files_meta["fully_labeled"], "site"].unique().tolist()),
}
cmp_tbl = pd.DataFrame({"this run": {k: summary[k] for k in REFERENCE}, "EoS.9 log": REFERENCE})
cmp_tbl["match"] = cmp_tbl["this run"] == cmp_tbl["EoS.9 log"]
display(cmp_tbl)

with open(OUT_DIR / "training_data_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
class_stats.to_csv(OUT_DIR / "class_stats_trusted_windows.csv")
sc.drop(columns=["label_list"]).to_csv(OUT_DIR / "soundscape_windows.csv", index=False)
print("Wrote:", sorted(p.name for p in OUT_DIR.glob("*") if p.suffix in {".json", ".csv"}))

## Notes

* **Small labelled set.** In the EoS.9 logs the sequence models are trained on 59 fully labelled files, and only 71 of 234 classes have a positive in them. The other classes rely on Perch logits, proxies, priors and the SED.
* **Training-set metrics.** In EoS.9 the ResidualSSM correction weight and the per-class thresholds are chosen from predictions on the training windows, so the near-perfect AUC in its logs is not a test estimate.
* **Better validation.** For an honest estimate, hold out whole files or sites (see the grouped folds in section 5) and tune on those.